# BERT-BiLSTM Marine Incident Classifier

Notebook version of the BERT + BiLSTM text-classification architecture from:

> Zhao, Z.; Liu, X.; Feng, L.; Grifoll, M.; Feng, H. (2025). *Causation
> Analysis of Marine Traffic Accidents Using Deep Learning Approaches: A
> Case Study from China's Coasts.* Systems, 13(4), 284.
> https://doi.org/10.3390/systems13040284

applied to [`baker-street/maib-incident-reports-5K`](https://huggingface.co/datasets/baker-street/maib-incident-reports-5K),
a dataset of UK MAIB marine incident narratives labeled by incident type
(Collision, Grounding/Stranding, Fire/Explosion, Accident to person(s),
Damage/Loss of Equipment, ...).

**Architecture (Section 3.1 / Figure 1 of the paper):** BERT encoder
(contextual token embeddings) -> Dropout -> 128-unit Bidirectional LSTM
(sequential dependencies over the tokens) -> concat(final forward, final
backward hidden states) -> Mish activation -> Dropout -> Linear -> softmax
over classes.

**Hyperparameters** are taken directly from Table 9 ("Model parameter
settings") of the paper for its BERT + BiLSTM configuration, and from
Section 4.1 (train/val/test split) and Figure 7 (epoch count to
convergence):

| Hyperparameter | Paper value | Source |
|---|---|---|
| BiLSTM hidden size | 128 | Table 9 |
| Learning rate | 1e-6 (single rate, AdamW) | Table 9 |
| L2 weight decay | 0.05 | Table 9 |
| L1 regularization | 5e-10 | Table 9 |
| Gradient clip (max norm) | 2.75 | Table 9 |
| Batch size | 32 | Table 9 |
| Dropout layers | 2 (bracketing the BiLSTM) | Table 9 |
| Activation (BiLSTM branch) | Mish | Table 9 |
| Epochs to convergence | ~20 | Figure 7 caption |
| Train / val / test split | 70% / 15% / 15% | Section 4.1 |

**Caveat:** the paper tuned these on its own dataset -- ~25,930 examples
across 32 causal-factor classes, sourced from Chinese and international
(GISIS) accident reports, class-balanced with text augmentation. The MAIB
dataset used here is ~5.8k examples across a handful of *incident-type*
classes (not causal factors), with no augmentation. The architecture and
hyperparameters are reproduced faithfully, but expect different absolute
accuracy on this smaller, differently-labeled dataset. The paper's
subsequent Apriori association-rule mining stage (which operates on
causal-factor tags, not incident types) is not implemented here.

## 1. Setup

In [ ]:
!pip install -q torch transformers datasets scikit-learn

In [ ]:
import json
import random

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import BertModel, BertTokenizerFast

## 2. Config

Defaults match Table 9 / Section 4.1 of the paper (see table above). Adjust as needed.

In [ ]:
class Config:
    dataset_name = "baker-street/maib-incident-reports-5K"
    bert_name = "bert-base-uncased"
    max_length = 128

    lstm_hidden = 128       # Table 9: "LSTM layer 128"
    lstm_layers = 1
    dropout = 0.3
    freeze_bert = False

    batch_size = 32         # Table 9
    epochs = 20             # Figure 7: converges by ~20 epochs
    lr = 1e-6               # Table 9 (single rate for all parameters)
    l2_weight_decay = 0.05  # Table 9 "L2 Regularization"
    l1_lambda = 5e-10       # Table 9 "L1 Regularization"
    max_grad_norm = 2.75    # Table 9 "Gradient"

    val_size = 0.15         # Section 4.1: 70/15/15 split
    test_size = 0.15
    seed = 42


cfg = Config()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 3. Load and split the dataset

The dataset ships a single `train` split, so we carve stratified train/val/test splits out of it (70/15/15, per Section 4.1).

In [ ]:
import re


def clean_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


raw = load_dataset(cfg.dataset_name, split="train")
texts = [clean_text(t) for t in raw["text"]]
labels_raw = raw["label"]

label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels_raw)
num_classes = len(label_encoder.classes_)
print(f"{len(texts)} examples, {num_classes} classes:", list(label_encoder.classes_))

In [ ]:
holdout_size = cfg.val_size + cfg.test_size
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=holdout_size, random_state=cfg.seed, stratify=labels
)
relative_test_size = cfg.test_size / holdout_size
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=relative_test_size, random_state=cfg.seed, stratify=temp_labels
)

print("train:", len(train_texts), "val:", len(val_texts), "test:", len(test_texts))

## 4. Dataset and tokenizer

In [ ]:
class MAIBTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


tokenizer = BertTokenizerFast.from_pretrained(cfg.bert_name)

train_ds = MAIBTextDataset(train_texts, train_labels, tokenizer, cfg.max_length)
val_ds = MAIBTextDataset(val_texts, val_labels, tokenizer, cfg.max_length)
test_ds = MAIBTextDataset(test_texts, test_labels, tokenizer, cfg.max_length)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)

## 5. Model: BERT + BiLSTM classifier

Matches Figure 1 / Table 9 of the paper: BERT -> dropout -> 128-unit BiLSTM -> concat final forward/backward states -> Mish -> dropout -> linear -> softmax.

In [ ]:
class BertBiLSTMClassifier(nn.Module):
    def __init__(
        self,
        num_classes,
        bert_name="bert-base-uncased",
        lstm_hidden=128,
        lstm_layers=1,
        dropout=0.3,
        freeze_bert=False,
    ):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_name)
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        bert_hidden = self.bert.config.hidden_size
        self.bilstm = nn.LSTM(
            input_size=bert_hidden,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        # Two dropout layers bracketing the BiLSTM, per Table 9 ("Dropout Layer: 2").
        self.dropout_pre_lstm = nn.Dropout(dropout)
        self.dropout_post_lstm = nn.Dropout(dropout)
        self.mish = nn.Mish()
        self.classifier = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        sequence_output = self.bert(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state  # (batch, seq_len, bert_hidden)
        sequence_output = self.dropout_pre_lstm(sequence_output)

        _, (h_n, _) = self.bilstm(sequence_output)
        # h_n: (num_layers * 2, batch, lstm_hidden); last layer's forward/backward states
        forward_hidden = h_n[-2]
        backward_hidden = h_n[-1]
        pooled = torch.cat([forward_hidden, backward_hidden], dim=1)

        pooled = self.mish(pooled)
        pooled = self.dropout_post_lstm(pooled)
        return self.classifier(pooled)


model = BertBiLSTMClassifier(
    num_classes=num_classes,
    bert_name=cfg.bert_name,
    lstm_hidden=cfg.lstm_hidden,
    lstm_layers=cfg.lstm_layers,
    dropout=cfg.dropout,
    freeze_bert=cfg.freeze_bert,
).to(device)

sum(p.numel() for p in model.parameters() if p.requires_grad), "trainable parameters"

## 6. Optimizer

A single learning rate across all parameters, with AdamW's decoupled weight decay as the L2 term -- matching Table 9. The L1 term is added to the loss manually inside the training loop, since AdamW has no native L1 option.

In [ ]:
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.l2_weight_decay)
loss_fn = nn.CrossEntropyLoss()

## 7. Training loop

In [ ]:
def run_epoch(model, loader, optimizer=None, max_grad_norm=2.75, l1_lambda=0.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(is_train):
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            batch_labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, batch_labels)

            if is_train:
                if l1_lambda > 0:
                    l1_penalty = sum(p.abs().sum() for p in model.parameters())
                    loss = loss + l1_lambda * l1_penalty

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()

            total_loss += loss.item() * batch_labels.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(batch_labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, macro_f1, all_preds, all_labels

In [ ]:
best_val_f1 = -1.0
best_state = None

for epoch in range(1, cfg.epochs + 1):
    train_loss, train_acc, train_f1, _, _ = run_epoch(
        model, train_loader, optimizer, cfg.max_grad_norm, cfg.l1_lambda
    )
    val_loss, val_acc, val_f1, _, _ = run_epoch(model, val_loader)

    print(
        f"Epoch {epoch}/{cfg.epochs} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_f1={train_f1:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  -> new best (val_f1={val_f1:.4f})")

model.load_state_dict(best_state)

## 8. Test evaluation

In [ ]:
test_loss, test_acc, test_f1, test_preds, test_labels_ = run_epoch(model, test_loader)
print(f"Test: loss={test_loss:.4f} acc={test_acc:.4f} macro_f1={test_f1:.4f}\n")
print(
    classification_report(
        test_labels_, test_preds, target_names=label_encoder.classes_, digits=4
    )
)

## 9. Save checkpoint

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/best_model.pt")
with open("checkpoints/label_classes.json", "w") as f:
    json.dump(list(label_encoder.classes_), f)
print("Saved checkpoints/best_model.pt and checkpoints/label_classes.json")

## 10. Inference on new narratives

In [ ]:
def predict(text, model=model, tokenizer=tokenizer, classes=label_encoder.classes_):
    model.eval()
    encoding = tokenizer(
        clean_text(text),
        truncation=True,
        padding="max_length",
        max_length=cfg.max_length,
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1).squeeze(0)
        pred_idx = int(probs.argmax().item())

    return classes[pred_idx], float(probs[pred_idx])


text = "A bulk carrier ran aground after losing steering control in heavy weather."
label, confidence = predict(text)
print(f"Text: {text}\nPredicted: {label} (confidence={confidence:.3f})")